<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.5.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 7.5.1**
> A notebook to help you get started  
> DSI DSSG + MNPS   

> # **Version 7.5.1 Change**
> - **Back to v7.5 Foundation**: Uses the proven v7.5 correction approach
> - **GPT-4o-2024-11-20**: Stable model with reliable performance
> - **Zero-Shot Prompt**: Includes the essential prompt from v7.1
> - **Comprehensive MNPS Resources**: All 4 critical documents from MNPS Prompt Resources.zip
> - **Attribute-Only Evaluation**: Ignores job titles, focuses on job attributes
> - **v7.5 Corrections**: Applies the proven correction logic at the end


In [2]:
# ==== 1) Imports, paths, inputs from v7.1 artifacts ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Run folder: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251014_202724
📁 Outputs dir: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251014_202724/outputs
📦 Found /content/MNPS Prompt Resources.zip, extracting...
✅ Extracted MNPS Prompt Resources
📄 Batch input: /content/Sample JDs.csv
📄 Ground truth: /content/Ground Truth Masterfile.csv
📄 MNPS roles: /content/MNPS Roles.csv
📄 MNPS KSACs: /content/MNPS KSACs.csv
📄 Competency Extended: /content/Competency Extended Descriptions.csv
📄 Korn Ferry: /content/Korn_Ferry Lominger 38 Competencies.csv
✅ Loaded 43 job descriptions
✅ Loaded 176 ground truth records
✅ Loaded 60 MNPS roles
✅ Loaded 300 MNPS KSACs
✅ Loaded 38 competency descriptions
✅ Loaded 38 Korn Ferry competencies


In [3]:
# ==== 2) Load data and build attribute-only view (ignore title) ====

# Load prediction data (if available from previous runs)
# For now, we'll work with the raw job descriptions
preds = df.copy()

# Build attribute-only text (ignore job titles)
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]

# Combine all attribute text
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']

print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")


✅ Built attribute-only view for 43 job descriptions
✅ Ignoring job titles - focusing on job attributes only


In [4]:
# ==== 3) Closed sets and normalization helpers ====

# Get MNPS roles from the loaded data
# Handle different possible column names
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    VALID_ROLES = roles_df[role_columns[0]].dropna().tolist()
else:
    # Fallback to first column
    VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")

# Closed sets for major and minor role groups
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver'
]

MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']

# Normalization mapping for minor roles
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

# Specialist fallback patterns
SPECIALIST_FALLBACKS = [
    ('Teacher', 'classroom|lesson|instruction|teacher|students'),
    ('Coach', 'coach|instructional coach|plc|model lessons|co-teach'),
    ('Clerical Support', 'clerk|clerical|records|data entry|office support'),
    ('Counselor', 'counsel|social-emotional|guidance'),
    ('Manager', 'manage|supervise|budget|oversight|lead team|program manager'),
]

def normalize_minor(x: str) -> str:
    """Normalize minor role to approved values."""
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    """Discourage overuse of 'Specialist' based on attribute content."""
    if proposed_major != 'Specialist':
        return proposed_major
    t = (text or '').lower()
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    return proposed_major

print("✅ Closed sets and normalization helpers defined")


✅ Found 60 MNPS roles
✅ Closed sets and normalization helpers defined


In [5]:
# ==== 4) Build comprehensive KSACs text from all MNPS resources ====

def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n\n"

    # Clean up column names to handle potential whitespace or case issues
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()

    # Add role-specific KSACs
    # Find columns that contain 'Role' and 'KSACs' (case-insensitive and partial match)
    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)

    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n\n"
    else:
        print("Warning: Could not find 'Role' or 'KSACs' columns in ksacs_df.")

    # Add competency extended descriptions
    # Find columns that contain 'Competency' and 'Description' (case-insensitive and partial match)
    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)

    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description' columns in competency_df.")

    # Add Korn Ferry competencies
    # Find columns that contain 'Competency' and 'Definition' (case-insensitive and partial match)
    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)

    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description'/'Definition' columns in korn_ferry_df.")

    return ksacs_text

KSACS_TEXT = build_ksacs_text()

print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")
print("✅ Includes all 4 critical MNPS resource documents")


✅ Built comprehensive KSACs text (37794 characters)
✅ Includes all 4 critical MNPS resource documents


In [6]:
# ==== 5) Zero Shot Prompt (from v7.1) ====
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.

- Group jobs that have similar functions, responsibilities, and requirements, regardless of their job titles.

- Use the attached reference sources (Ground Truth Masterfile, MNPS Roles, MNPS KSACs) to ensure alignment with MNPS standards and classifications.

- Focus on the actual work being performed, not the job title, to create meaningful and accurate groupings.

- Ensure that each grouping reflects the true nature of the work and aligns with MNPS role classifications and competency frameworks.

- Provide clear justification for each grouping decision based on the job attributes and MNPS standards.

- Never justify classifications based on job titles - only use job attributes and MNPS standards.

Output Requirements:
- Major Role Group: Choose from approved MNPS major role groupings
- Minor Sub Group: Use I, II, III, or Lead based on complexity and responsibility level
- Provide detailed justification based on job attributes and MNPS KSACs alignment"""

print("✅ Zero shot prompt defined")


✅ Zero shot prompt defined


In [7]:
# ==== 6) OpenAI API Setup ====
import os
from google.colab import userdata

# Get API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

# Use GPT-4o-2024-11-20 for stable performance
MODEL_ID = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")

def call_llm_json(prompt: str, model: str = None) -> dict:
    """Call OpenAI API with JSON response."""
    if model is None:
        model = MODEL_ID

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.2
    )
    return json.loads(response.choices[0].message.content)

print("✅ call_llm_json function defined")


✅ OpenAI client initialized
✅ Using model: gpt-4o-2024-11-20
✅ call_llm_json function defined


In [8]:
# ==== 7) Batch Processing with GPT-4o ====
import time
from tqdm import tqdm

def process_job_description(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description using GPT-4o."""
    # Build job description text (ignore job title)
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""

    # Build comprehensive prompt
    prompt = f"""{zero_shot_prompt}

Available MNPS Roles: {', '.join(VALID_ROLES)}

{KSACS_TEXT}

Job Description to Classify:
{job_text}

**IMPORTANT**:
- Ignore the job title completely
- Base classification solely on job attributes
- Use only approved MNPS roles and levels (I, II, III, Lead)
- Provide detailed justification based on KSACs alignment

Return your response as a JSON object with the following structure:
{{
  "new_job_title": "Descriptive title based on function and level",
  "major_role_group": "One of the approved MNPS roles",
  "minor_sub_group": "I, II, III, or Lead",
  "grouping_justification": "Detailed explanation based on job attributes and KSACs alignment"
}}"""

    try:
        # Use GPT-4o for classification
        response_data = call_llm_json(prompt, MODEL_ID)

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': response_data.get('new_job_title', 'Unknown'),
            'major_role_group': response_data.get('major_role_group', 'Other'),
            'minor_sub_group': response_data.get('minor_sub_group', 'I'),
            'grouping_justification': response_data.get('grouping_justification', 'No justification provided'),
            'model_used': MODEL_ID
        }
    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions
results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    time.sleep(0.1)  # Rate limiting

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v75.csv"
results_df.to_csv(output_path, index=False)

print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")


Processing jobs: 100%|██████████| 43/43 [10:51<00:00, 15.16s/it]

✅ Processed 43 job descriptions
✅ Saved results to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251014_202724/outputs/Job_Classifications_Batch_gpt4o_v75.csv


In [9]:
# ==== 8) Apply v7.5 Corrections ====

# Load the GPT-4o results
preds = results_df.copy()

# Apply corrections
maj0 = preds.get('major_role_group', pd.Series(['Other']*len(preds)))
min0 = preds.get('minor_sub_group', pd.Series(['I']*len(preds)))

ref_major = []
for i, m in enumerate(maj0):
    proposed = str(m) if pd.notna(m) else 'Other'
    proposed = proposed if proposed in MAJOR_ALLOWED else 'Other'
    proposed = discourage_specialist(text.iloc[i], proposed)
    ref_major.append(proposed)

ref_minor = [normalize_minor(x) for x in min0]

# Create corrected results
corrected = preds.copy()
corrected['major_role_group'] = ref_major
corrected['minor_sub_group'] = ref_minor

# Save corrected results
corrected_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v75_corrected.csv"
corrected.to_csv(corrected_path, index=False)

print(f"✅ Applied v7.5 corrections")
print(f"✅ Saved corrected results to: {corrected_path}")


✅ Applied v7.5 corrections
✅ Saved corrected results to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251014_202724/outputs/Job_Classifications_Batch_gpt4o_v75_corrected.csv


In [10]:
# ==== 9) Generate Summary Statistics and Examples ====

before_major = preds.get('major_role_group', pd.Series(['']*len(preds))).astype(str)
before_minor = preds.get('minor_sub_group', pd.Series(['']*len(preds))).astype(str)
after_major  = corrected['major_role_group'].astype(str)
after_minor  = corrected['minor_sub_group'].astype(str)

counts = pd.DataFrame({
    'key': ['rows','major_changed','minor_changed','specialist_after_count'],
    'value': [
        len(corrected),
        int((before_major!=after_major).sum()),
        int((before_minor!=after_minor).sum()),
        int((after_major=='Specialist').sum())
    ]
})

counts_path = OUTPUTS_DIR / "correction_counts_gpt4o_v75.csv"
counts.to_csv(counts_path, index=False)

# Show examples of changes
ex_idx = ((before_major!=after_major) | (before_minor!=after_minor)).to_numpy().nonzero()[0][:6]
examples = pd.DataFrame({
    'row': ex_idx,
    'job_title_original': preds['job_title_original'].iloc[ex_idx],
    'major_before': before_major.iloc[ex_idx],
    'major_after': after_major.iloc[ex_idx],
    'minor_before': before_minor.iloc[ex_idx],
    'minor_after': after_minor.iloc[ex_idx],
})

examples_path = OUTPUTS_DIR / "examples_gpt4o_v75.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 Summary Statistics:")
print(counts.to_string(index=False))

print("\n📝 Example Corrections:")
print(examples.to_string(index=False))

print(f"\n✅ Saved counts to: {counts_path}")
print(f"✅ Saved examples to: {examples_path}")



📊 Summary Statistics:
                   key  value
                  rows     43
         major_changed     21
         minor_changed      1
specialist_after_count      0

📝 Example Corrections:
 row job_title_original major_before major_after minor_before minor_after
   0                      Supervisor       Other         Lead        Lead
   1                       Therapist       Other            I           I
   4                      Specialist     Teacher            I           I
   5                       Principal       Other         Lead        Lead
   7                      Specialist     Teacher            I           I
  10                       Librarian       Other         Lead        Lead

✅ Saved counts to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251014_202724/outputs/correction_counts_gpt4o_v75.csv
✅ Saved examples to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251014_202724/outputs/examples_gpt4o_v75.csv
